# **MULTILINEAR TENSOR REGRESSION FOR LONGITUDINAL RELATIONAL DATA** (Peter Hoff, 2015)

Cargar datos

In [ ]:

import numpy as np
import json


# ============================================================
# DATOS PARA EL MODELO
# ============================================================
X_flat = np.load("data/X.npy")
X = X_flat.reshape((25, 25, 4, 3, 2, 543),order="F")
X_1w = np.load("data/X_1w.npy")
X_1m = np.load("data/X_1m.npy")
Y_flat = np.load("data/Y.npy")
Y = Y_flat.reshape((25, 25, 4, 543), order="F")


# ============================================================
# COMPROBACIONES
# ============================================================
print("X:", X.shape)
print("X_1w:", X_1w.shape)
print("X_1m:", X_1m.shape)
print("Y   :", Y.shape)

assert X_1w.shape == (25, 25, 12, 543)
assert X_1m.shape == (25, 25, 12, 543)
assert Y.shape == (25, 25, 4, 543)


# ============================================================
# CARGAR NOMBRES DE LAS DIMENSIONES
# ============================================================

with open("data/dimnames_X.json", "r") as f:
    dimnames_X = json.load(f)

with open("data/dimnames_X_1w.json", "r") as f:
    dimnames_X_1w = json.load(f)

with open("data/dimnames_X_1m.json", "r") as f:
    dimnames_X_1m = json.load(f)

with open("data/dimnames_Y.json", "r") as f:
    dimnames_Y = json.load(f)


# ============================================================
# MOSTRAR NOMBRES
# ============================================================
print("\nDimensiones X:")

for k, names in enumerate(dimnames_X):
    print(
        f"Modo {k}: "
        f"{len(names)} elementos -> "
        f"{names[:5]}"
    )



print("\nDimensiones X_1w:")

for k, names in enumerate(dimnames_X_1w):
    print(
        f"Modo {k}: "
        f"{len(names)} elementos -> "
        f"{names[:5]}"
    )


print("\nDimensiones X_1m:")

for k, names in enumerate(dimnames_X_1m):
    print(
        f"Modo {k}: "
        f"{len(names)} elementos -> "
        f"{names[:5]}"
    )


print("\nDimensiones Y:")

for k, names in enumerate(dimnames_Y):
    print(
        f"Modo {k}: "
        f"{len(names)} elementos -> "
        f"{names[:5]}"
    )


X: (25, 25, 4, 3, 2, 543)
X_1w: (25, 25, 12, 543)
X_1m: (25, 25, 12, 543)
Y   : (25, 25, 4, 543)

Dimensiones X:
Modo 0: 25 elementos -> ['AFG', 'AUS', 'CHN', 'DEU', 'EGY']
Modo 1: 25 elementos -> ['AFG', 'AUS', 'CHN', 'DEU', 'EGY']
Modo 2: 4 elementos -> ['mn', 'mp', 'vn', 'vp']
Modo 3: 3 elementos -> ['ij', 'ji', 'ijk']
Modo 4: 2 elementos -> ['1w', '1m']
Modo 5: 543 elementos -> ['20040205', '20040212', '20040219', '20040226', '20040304']

Dimensiones X_1w:
Modo 0: 25 elementos -> ['AFG', 'AUS', 'CHN', 'DEU', 'EGY']
Modo 1: 25 elementos -> ['AFG', 'AUS', 'CHN', 'DEU', 'EGY']
Modo 2: 12 elementos -> ['mn-ij', 'mn-ji', 'mn-ijk', 'mp-ij', 'mp-ji']
Modo 3: 543 elementos -> ['20040205', '20040212', '20040219', '20040226', '20040304']

Dimensiones X_1m:
Modo 0: 25 elementos -> ['AFG', 'AUS', 'CHN', 'DEU', 'EGY']
Modo 1: 25 elementos -> ['AFG', 'AUS', 'CHN', 'DEU', 'EGY']
Modo 2: 12 elementos -> ['mn-ij', 'mn-ji', 'mn-ijk', 'mp-ij', 'mp-ji']
Modo 3: 543 elementos -> ['20040205', '20040212'

Funciones auxiliares para operaciones entre tensores

In [ ]:

import numpy as np


# ============================================================
# 1. MATRICIZATION
# ============================================================

def mat(A, k):
    """
    Matricización de un tensor a lo largo del modo k.

    Equivalente a la función mat() de tfunctions.r de Peter Hoff.

    Parameters
    ----------
    A : np.ndarray
        Tensor de dimensión (d1, d2, ..., dK).
    k : int
        Modo a matricizar. Python usa índices 0-based.

    Returns
    -------
    A_k : np.ndarray
        Matriz con:
            filas    = dimensión del modo k
            columnas = producto de las demás dimensiones.

    Example
    -------
    A.shape = (3, 4, 5)

    mat(A, 0).shape = (3, 20)
    mat(A, 1).shape = (4, 15)
    mat(A, 2).shape = (5, 12)
    """

    if not isinstance(A, np.ndarray):
        A = np.asarray(A)

    K = A.ndim

    if k < 0 or k >= K:
        raise ValueError(
            f"El modo k={k} no existe para un tensor "
            f"con {K} dimensiones."
        )

    # Movemos el modo k al frente.
    A_perm = np.moveaxis(A, k, 0)

    # Primera dimensión = modo k.
    # Las demás dimensiones se convierten en columnas.
    A_k = A_perm.reshape(
        A.shape[k],
        -1,
        order="F"
    )

    return A_k


# ============================================================
# 2. ARRAY-MATRIX PRODUCT
# ============================================================

def amprod(A, M, k):
    """
    Producto de un tensor A por una matriz M a lo largo
    del modo k.

    Equivalente a amprod() de tfunctions.r.

    Matemáticamente:

        A ×_k M

    donde:

        A.shape[k] = M.shape[1]

    y el resultado tiene:

        result.shape[k] = M.shape[0]

    Parameters
    ----------
    A : np.ndarray
        Tensor de entrada.

    M : np.ndarray
        Matriz que multiplica el modo k.

    k : int
        Modo sobre el cual se realiza la multiplicación.
        Python usa índices 0-based.

    Returns
    -------
    result : np.ndarray
        Tensor transformado.

    Example
    -------
    A.shape = (5, 4, 3)
    M.shape = (2, 5)

    amprod(A, M, 0).shape = (2, 4, 3)
    """

    A = np.asarray(A)
    M = np.asarray(M)

    if A.ndim == 0:
        raise ValueError("A debe ser un tensor.")

    if M.ndim != 2:
        raise ValueError("M debe ser una matriz 2D.")

    if k < 0 or k >= A.ndim:
        raise ValueError(
            f"El modo k={k} no existe para A.ndim={A.ndim}."
        )

    if M.shape[1] != A.shape[k]:
        raise ValueError(
            f"Dimensiones incompatibles: "
            f"M tiene {M.shape[1]} columnas, "
            f"pero A tiene dimensión {A.shape[k]} "
            f"en el modo {k}."
        )

    # Matricización de A en el modo k
    A_k = mat(A, k)

    # Multiplicación matricial
    # M:    (new_dim, old_dim)
    # A_k:  (old_dim, resto)
    #
    # MA:   (new_dim, resto)
    MA = M @ A_k

    # Nuevas dimensiones:
    # sustituimos solamente la dimensión k
    new_shape = list(A.shape)
    new_shape[k] = M.shape[0]

    # Primero reconstruimos colocando el modo k al frente.
    MA_front = MA.reshape(
        [M.shape[0]] + list(A.shape[:k]) + list(A.shape[k + 1:]),
        order="F"
    )

    # Actualmente:
    #
    # (new_dim, dimensiones antes de k, dimensiones después de k)
    #
    # Queremos:
    #
    # (dimensiones antes de k, new_dim, dimensiones después de k)
    #
    # Para eso movemos el eje 0 nuevamente a la posición k.
    result = np.moveaxis(MA_front, 0, k)

    return result


# ============================================================
# 3. TUCKER PRODUCT
# ============================================================

def tprod(A, B, modes=None):
    """
    Tucker product de un tensor A por una lista de matrices B.

    Equivalente a tprod() de tfunctions.r de Peter Hoff.

    Si:

        B = [B1, B2, ..., BK]

    entonces:

        tprod(A, B)
        =
        A ×_1 B1 ×_2 B2 ... ×_K BK

    Parameters
    ----------
    A : np.ndarray
        Tensor de entrada.

    B : list[np.ndarray]
        Lista de matrices.

    modes : iterable of int, optional
        Modos sobre los cuales realizar la multiplicación.
        Si es None, se utilizan todos los modos correspondientes
        a B.

    Returns
    -------
    X : np.ndarray
        Tucker product.

    Example
    -------
    A.shape = (5, 4, 3)

    B[0].shape = (2, 5)
    B[1].shape = (6, 4)
    B[2].shape = (7, 3)

    tprod(A, B).shape = (2, 6, 7)
    """

    if not isinstance(B, (list, tuple)):
        raise TypeError("B debe ser una lista o tupla de matrices.")

    if modes is None:
        modes = range(len(B))
    else:
        modes = list(modes)

    X = np.asarray(A)

    for k in modes:

        if k < 0 or k >= X.ndim:
            raise ValueError(
                f"El modo {k} no existe para un tensor "
                f"con {X.ndim} dimensiones."
            )

        X = amprod(X, B[k], k)

    return X


# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

def tensor_shape_info(X, name="X"):
    """
    Imprime información básica sobre las dimensiones del tensor.
    """

    print(f"{name}.shape = {X.shape}")
    print(f"{name}.ndim  = {X.ndim}")

    for k, dim in enumerate(X.shape):
        print(f"  modo {k}: {dim}")


def check_tprod_dimensions(X, B, modes=None):
    """
    Comprueba que las dimensiones de las matrices B sean
    compatibles con X para un Tucker product.

    No realiza el producto.
    """

    if modes is None:
        modes = range(len(B))

    for k in modes:

        if k >= X.ndim:
            raise ValueError(
                f"El modo {k} no existe en X."
            )

        expected = X.shape[k]
        actual = B[k].shape[1]

        if expected != actual:
            raise ValueError(
                f"B[{k}] incompatible: "
                f"X.shape[{k}] = {expected}, "
                f"pero B[{k}].shape[1] = {actual}."
            )

    return True

Validación de operaciones entre tensores

In [ ]:

import numpy as np


# ============================================================
# CONFIGURACIÓN
# ============================================================

np.set_printoptions(
    precision=4,
    suppress=True
)

rng = np.random.default_rng(12345)


# ============================================================
# FUNCIÓN AUXILIAR PARA LAS PRUEBAS
# ============================================================

def check(condition, message):
    """
    Imprime PASS/FAIL y detiene la ejecución si falla.
    """

    if condition:
        print(f"PASS: {message}")
    else:
        print(f"FAIL: {message}")
        raise AssertionError(message)


# ============================================================
# 1. PRUEBA DE MAT()
# ============================================================

print("\n")
print("=" * 70)
print("1. PRUEBA DE MATRICIZACIÓN")
print("=" * 70)


# Tensor pequeño conocido.
#
# Dimensiones:
#
# A = 2 × 3 × 4

A = np.arange(1, 25).reshape(
    (2, 3, 4),
    order="F"
)

print("\nTensor A:")
print("shape:", A.shape)
print(A)


# ------------------------------------------------------------
# Matricización modo 0
# ------------------------------------------------------------

A0 = mat(A, 0)

print("\nmat(A, 0)")
print("shape:", A0.shape)
print(A0)


# ------------------------------------------------------------
# Matricización modo 1
# ------------------------------------------------------------

A1 = mat(A, 1)

print("\nmat(A, 1)")
print("shape:", A1.shape)
print(A1)


# ------------------------------------------------------------
# Matricización modo 2
# ------------------------------------------------------------

A2 = mat(A, 2)

print("\nmat(A, 2)")
print("shape:", A2.shape)
print(A2)


# Verificar dimensiones

check(
    A0.shape == (2, 12),
    "mat(A, 0) tiene dimensión (2,12)"
)

check(
    A1.shape == (3, 8),
    "mat(A, 1) tiene dimensión (3,8)"
)

check(
    A2.shape == (4, 6),
    "mat(A, 2) tiene dimensión (4,6)"
)


# ============================================================
# 2. PRUEBA DE MAT CON EL TENSOR REAL
# ============================================================

print("\n")
print("=" * 70)
print("2. MATRICIZACIÓN DE X_1w")
print("=" * 70)


X_1w = np.load(
    "data/X_1w.npy"
)

print("\nX_1w:")
tensor_shape_info(
    X_1w,
    "X_1w"
)


# X_1w:
#
# (25, 25, 12, 543)
#
# Para Hoff:
#
# modo 0 = países
# modo 1 = países
# modo 2 = relación × medición
# modo 3 = tiempo


for k in range(X_1w.ndim):

    Xk = mat(X_1w, k)

    expected_shape = (
        X_1w.shape[k],
        X_1w.size // X_1w.shape[k]
    )

    print(
        f"\nmat(X_1w,{k})"
    )

    print(
        "shape:",
        Xk.shape
    )

    print(
        "shape esperado:",
        expected_shape
    )

    check(
        Xk.shape == expected_shape,
        f"mat(X_1w,{k}) tiene dimensiones correctas"
    )


# ============================================================
# 3. PRUEBA DE AMPROD CON UN EJEMPLO PEQUEÑO
# ============================================================

print("\n")
print("=" * 70)
print("3. PRUEBA DE AMPROD")
print("=" * 70)


# Tensor:
#
# A : 2 × 3 × 4
#
# Matriz:
#
# M : 5 × 2
#
# Entonces:
#
# A ×_0 M
#
# debe tener:
#
# 5 × 3 × 4

A = rng.normal(
    size=(2, 3, 4)
)

M = rng.normal(
    size=(5, 2)
)

AM0 = amprod(
    A,
    M,
    0
)

print("\nA.shape =", A.shape)
print("M.shape =", M.shape)
print("amprod(A,M,0).shape =", AM0.shape)


check(
    AM0.shape == (5, 3, 4),
    "amprod en modo 0 produce (5,3,4)"
)


# ------------------------------------------------------------
# Modo 1
# ------------------------------------------------------------

M1 = rng.normal(
    size=(6, 3)
)

AM1 = amprod(
    A,
    M1,
    1
)

print("\nM1.shape =", M1.shape)
print("amprod(A,M1,1).shape =", AM1.shape)

check(
    AM1.shape == (2, 6, 4),
    "amprod en modo 1 produce (2,6,4)"
)


# ------------------------------------------------------------
# Modo 2
# ------------------------------------------------------------

M2 = rng.normal(
    size=(7, 4)
)

AM2 = amprod(
    A,
    M2,
    2
)

print("\nM2.shape =", M2.shape)
print("amprod(A,M2,2).shape =", AM2.shape)

check(
    AM2.shape == (2, 3, 7),
    "amprod en modo 2 produce (2,3,7)"
)


# ============================================================
# 4. COMPROBAR AMPROD CONTRA LA DEFINICIÓN MATRICIAL
# ============================================================

print("\n")
print("=" * 70)
print("4. VALIDACIÓN MATEMÁTICA DE AMPROD")
print("=" * 70)


# Usamos nuevamente:
#
# A : 2 × 3 × 4
# M : 5 × 2
#
# La definición dice:
#
# mat(A ×_0 M, 0)
# =
# M @ mat(A, 0)

A = rng.normal(
    size=(2, 3, 4)
)

M = rng.normal(
    size=(5, 2)
)

AM = amprod(
    A,
    M,
    0
)

left = mat(
    AM,
    0
)

right = M @ mat(
    A,
    0
)


print("\nLado izquierdo:")
print(
    "mat(amprod(A,M,0),0).shape =",
    left.shape
)

print("\nLado derecho:")
print(
    "M @ mat(A,0).shape =",
    right.shape
)


difference = np.max(
    np.abs(left - right)
)

print(
    "\nMáxima diferencia:",
    difference
)


check(
    np.allclose(left, right),
    "amprod satisface la definición matricial"
)


# ============================================================
# 5. PRUEBA DE TUCKER PRODUCT
# ============================================================

print("\n")
print("=" * 70)
print("5. PRUEBA DE TUCKER PRODUCT")
print("=" * 70)


# Tensor:
#
# A : 2 × 3 × 4
#
# B0 : 5 × 2
# B1 : 6 × 3
# B2 : 7 × 4
#
# Resultado:
#
# 5 × 6 × 7

A = rng.normal(
    size=(2, 3, 4)
)

B = [

    rng.normal(size=(5, 2)),

    rng.normal(size=(6, 3)),

    rng.normal(size=(7, 4))

]


check_tprod_dimensions(
    A,
    B
)

ABC = tprod(
    A,
    B
)

print("\nA.shape =", A.shape)

for k, Bk in enumerate(B):

    print(
        f"B[{k}].shape =",
        Bk.shape
    )

print(
    "tprod(A,B).shape =",
    ABC.shape
)


check(
    ABC.shape == (5, 6, 7),
    "tprod produce las dimensiones esperadas"
)


# ============================================================
# 6. VALIDAR TUCKER PRODUCT PASO A PASO
# ============================================================

print("\n")
print("=" * 70)
print("6. TUCKER PRODUCT: PASO A PASO")
print("=" * 70)


step1 = amprod(
    A,
    B[0],
    0
)

step2 = amprod(
    step1,
    B[1],
    1
)

step3 = amprod(
    step2,
    B[2],
    2
)


print("\nDespués de B0:")
print(step1.shape)

print("\nDespués de B1:")
print(step2.shape)

print("\nDespués de B2:")
print(step3.shape)


check(
    np.allclose(ABC, step3),
    "tprod coincide con aplicar amprod secuencialmente"
)


# ============================================================
# 7. PRUEBA PARCIAL DE TPROD
# ============================================================

print("\n")
print("=" * 70)
print("7. TPROD SOBRE ALGUNOS MODOS")
print("=" * 70)


# Solo aplicamos B0 y B1.
#
# A : 2 × 3 × 4
#
# B0 : 5 × 2
# B1 : 6 × 3
#
# Resultado:
#
# 5 × 6 × 4

AB = tprod(
    A,
    B,
    modes=[0, 1]
)

print(
    "\ntprod(A,B,modes=[0,1]).shape =",
    AB.shape
)

check(
    AB.shape == (5, 6, 4),
    "tprod parcial produce (5,6,4)"
)


# ============================================================
# 8. PRUEBA DEL CASO IDENTITY
# ============================================================

print("\n")
print("=" * 70)
print("8. PRUEBA CON MATRICES IDENTIDAD")
print("=" * 70)


# Si:
#
# B0 = I
# B1 = I
# B2 = I
#
# entonces:
#
# tprod(A,B) = A

I = [

    np.eye(2),

    np.eye(3),

    np.eye(4)

]

A_identity = tprod(
    A,
    I
)

difference_identity = np.max(
    np.abs(A - A_identity)
)

print(
    "\nMáxima diferencia:",
    difference_identity
)

check(
    np.allclose(A, A_identity),
    "Tucker product con identidades devuelve A"
)


# ============================================================
# 9. PRUEBA CON X_1w
# ============================================================

print("\n")
print("=" * 70)
print("9. TUCKER PRODUCT CON X_1w")
print("=" * 70)


# X_1w:
#
# (25,25,12,543)
#
# Queremos transformar los primeros tres modos:
#
# B0 : 25 × 25
# B1 : 25 × 25
# B2 :  4 × 12
#
# El tiempo queda intacto:
#
# 543
#
# Por tanto:
#
# tprod(X_1w,B).shape
#
# = (25,25,4,543)

B_real = [

    rng.normal(
        size=(25, 25)
    ),

    rng.normal(
        size=(25, 25)
    ),

    rng.normal(
        size=(4, 12)
    )

]


check_tprod_dimensions(
    X_1w,
    B_real,
    modes=[0, 1, 2]
)


Y_hat_1w = tprod(
    X_1w,
    B_real,
    modes=[0, 1, 2]
)


print("\nX_1w.shape:")
print(X_1w.shape)

print("\nB0.shape:")
print(B_real[0].shape)

print("\nB1.shape:")
print(B_real[1].shape)

print("\nB2.shape:")
print(B_real[2].shape)

print("\nY_hat_1w.shape:")
print(Y_hat_1w.shape)


check(
    Y_hat_1w.shape == (25, 25, 4, 543),
    "tprod(X_1w,B) produce la dimensión de Y"
)


# ============================================================
# 10. VERIFICAR QUE EL TIEMPO NO FUE MODIFICADO
# ============================================================

print("\n")
print("=" * 70)
print("10. VERIFICAR MODO TEMPORAL")
print("=" * 70)


print(
    "X_1w.shape[-1] =",
    X_1w.shape[-1]
)

print(
    "Y_hat_1w.shape[-1] =",
    Y_hat_1w.shape[-1]
)


check(
    X_1w.shape[-1] == Y_hat_1w.shape[-1],
    "El modo temporal permanece en 543"
)


# ============================================================
# 11. PRUEBA CON X_1m
# ============================================================

print("\n")
print("=" * 70)
print("11. TUCKER PRODUCT CON X_1m")
print("=" * 70)


X_1m = np.load(
    "data/X_1m.npy"
)


Y_hat_1m = tprod(
    X_1m,
    B_real,
    modes=[0, 1, 2]
)


print(
    "X_1m.shape =",
    X_1m.shape
)

print(
    "Y_hat_1m.shape =",
    Y_hat_1m.shape
)


check(
    Y_hat_1m.shape == (25, 25, 4, 543),
    "tprod(X_1m,B) produce la dimensión de Y"
)


# ============================================================
# 12. PRUEBA FINAL: DOS ESCALAS TEMPORALES
# ============================================================

print("\n")
print("=" * 70)
print("12. DOS ESCALAS TEMPORALES")
print("=" * 70)


# Modelo:
#
# Y_hat =
#
#     tprod(X_1w, B_1w)
#
#     +
#
#     tprod(X_1m, B_1m)
#
#
# Cada término debe tener:
#
# (25,25,4,543)

B_1w = [

    rng.normal(
        size=(25, 25)
    ),

    rng.normal(
        size=(25, 25)
    ),

    rng.normal(
        size=(4, 12)
    )

]

B_1m = [

    rng.normal(
        size=(25, 25)
    ),

    rng.normal(
        size=(25, 25)
    ),

    rng.normal(
        size=(4, 12)
    )

]


Y_1w = tprod(
    X_1w,
    B_1w,
    modes=[0, 1, 2]
)

Y_1m = tprod(
    X_1m,
    B_1m,
    modes=[0, 1, 2]
)


Y_hat = Y_1w + Y_1m


print("\nY_1w.shape =", Y_1w.shape)
print("Y_1m.shape =", Y_1m.shape)
print("Y_hat.shape =", Y_hat.shape)


check(
    Y_1w.shape == (25,25,4,543),
    "Componente 1w tiene dimensión de Y"
)

check(
    Y_1m.shape == (25,25,4,543),
    "Componente 1m tiene dimensión de Y"
)

check(
    Y_hat.shape == (25,25,4,543),
    "La suma de las dos escalas tiene dimensión de Y"
)


# ============================================================
# RESULTADO FINAL
# ============================================================

print("\n")
print("=" * 70)
print("TODAS LAS PRUEBAS TERMINARON CORRECTAMENTE")
print("=" * 70)

print(
    """
Funciones verificadas:

    1. mat()
    2. amprod()
    3. tprod()
    4. tensor_shape_info()
    5. check_tprod_dimensions()

Y además:

    X_1w -> Tucker -> Y
    X_1m -> Tucker -> Y

Estructura:

    X_1w : (25,25,12,543)
    X_1m : (25,25,12,543)

    B_1w:
        (25,25)
        (25,25)
        (4,12)

    B_1m:
        (25,25)
        (25,25)
        (4,12)

    Y_hat:
        (25,25,4,543)
"""
)



1. PRUEBA DE MATRICIZACIÓN

Tensor A:
shape: (2, 3, 4)
[[[ 1  7 13 19]
  [ 3  9 15 21]
  [ 5 11 17 23]]

 [[ 2  8 14 20]
  [ 4 10 16 22]
  [ 6 12 18 24]]]

mat(A, 0)
shape: (2, 12)
[[ 1  3  5  7  9 11 13 15 17 19 21 23]
 [ 2  4  6  8 10 12 14 16 18 20 22 24]]

mat(A, 1)
shape: (3, 8)
[[ 1  2  7  8 13 14 19 20]
 [ 3  4  9 10 15 16 21 22]
 [ 5  6 11 12 17 18 23 24]]

mat(A, 2)
shape: (4, 6)
[[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]
 [13 14 15 16 17 18]
 [19 20 21 22 23 24]]
PASS: mat(A, 0) tiene dimensión (2,12)
PASS: mat(A, 1) tiene dimensión (3,8)
PASS: mat(A, 2) tiene dimensión (4,6)


2. MATRICIZACIÓN DE X_1w

X_1w:
X_1w.shape = (25, 25, 12, 543)
X_1w.ndim  = 4
  modo 0: 25
  modo 1: 25
  modo 2: 12
  modo 3: 543

mat(X_1w,0)
shape: (25, 162900)
shape esperado: (25, 162900)
PASS: mat(X_1w,0) tiene dimensiones correctas

mat(X_1w,1)
shape: (25, 162900)
shape esperado: (25, 162900)
PASS: mat(X_1w,1) tiene dimensiones correctas

mat(X_1w,2)
shape: (12, 339375)
shape esperado: (12, 339

In [16]:
import numpy as np


def initialize_B(m, p):
    """
    Inicialización de las matrices B_k siguiendo
    la lógica de mlm.ALS() de Peter Hoff.

    Parameters
    ----------
    m : tuple o list
        Dimensiones de Y.

        Ejemplo:
        m = (25, 25, 4, 543)

    p : tuple o list
        Dimensiones de X.

        Ejemplo:
        p = (25, 25, 12, 543)

    Returns
    -------
    B0 : list of np.ndarray
        Lista con las matrices iniciales B_k.

    Notes
    -----
    El último modo corresponde al tiempo y no tiene
    una matriz B asociada.

    Por tanto:

        K = len(m) - 1

    y solamente se generan B_0,...,B_{K-1}.
    """

    m = tuple(m)
    p = tuple(p)

    if len(m) != len(p):
        raise ValueError(
            "X e Y deben tener el mismo número de modos."
        )

    # El último modo es el temporal.
    K = len(m) - 1

    B0 = []

    for k in range(K):

        # Inicialización:
        #
        # B_k : m_k × p_k
        #
        # todos los elementos = 1/p_k

        Bk = np.full(
            (m[k], p[k]),
            1.0 / p[k]
        )

        # Si las dimensiones coinciden,
        # Hoff utiliza la identidad.
        if m[k] == p[k]:

            Bk = np.eye(
                m[k]
            )

        B0.append(Bk)

    return B0


m = Y.shape
p = X_1w.shape

B0 = initialize_B(m, p)

print("m =", m)
print("p =", p)

for k, Bk in enumerate(B0):

    print(
        f"B0[{k}].shape = {Bk.shape}"
    )

m = (25, 25, 4, 543)
p = (25, 25, 12, 543)
B0[0].shape = (25, 25)
B0[1].shape = (25, 25)
B0[2].shape = (4, 12)


In [22]:
# ============================================================
# MSE
# ============================================================

def mse(Y, EY):
    """
    Compute the mean squared error between observed and
    expected tensors.

    This reproduces the behavior of:

        mean((Y - EY)^2, na.rm = TRUE)

    in R.

    Parameters
    ----------
    Y : np.ndarray
        Observed response tensor.

    EY : np.ndarray
        Expected/fitted response tensor.

    Returns
    -------
    float
        Mean squared error.
    """

    Y = np.asarray(Y)
    EY = np.asarray(EY)

    if Y.shape != EY.shape:
        raise ValueError(
            f"Y and EY must have the same shape. "
            f"Got Y={Y.shape}, EY={EY.shape}."
        )

    return np.nanmean(
        (Y - EY) ** 2
    )


In [24]:
from tqdm import tqdm

def mlm_als(Y, X, tol=1e-5, imax=100, verbose=False, seed=None):
    """
    Alternating Least Squares estimation for multilinear
    tensor regression.
    This function is a Python translation of Peter Hoff's
    mlm.ALS() function.
    Model
    -----
        Y ≈ X ×_1 B_1 ×_2 B_2 ... ×_K B_K
    where the last dimension of X and Y is treated as the
    longitudinal/time dimension and is NOT transformed by
    a coefficient matrix.

    Parameters
    ----------
    Y : np.ndarray
        Response tensor.
        Example:
            Y.shape = (25, 25, 4, 543)
    X : np.ndarray
        Predictor tensor.
        Example:
            X.shape = (25, 25, 12, 543)
    tol : float, default=1e-5
        Convergence tolerance.
    imax : int, default=100
        Maximum number of ALS iterations.
    verbose : bool, default=False
        If True, print the MSE and relative improvement
        at every iteration.
    seed : int or None, default=None
        If None, use Hoff's deterministic initialization
        followed by initial least-squares estimates.
        If an integer is provided, initialize B randomly.
    Returns
    -------
    B : list of np.ndarray
        Estimated coefficient matrices.
        For the current problem:
            B[0].shape = (25,25)
            B[1].shape = (25,25)
            B[2].shape = (4,12)
    """
    # Convert inputs to NumPy arrays
    Y = np.asarray(Y, dtype=float)
    X = np.asarray(X, dtype=float)
    # Check that Y and X have the same number of dimensions.
    if Y.ndim != X.ndim:
        raise ValueError("X and Y must have the same number of dimensions.")
    # Check that the last dimension is the same.
    # The last dimension represents time/longitudinal
    # observations and is shared by X and Y.
    if Y.shape[-1] != X.shape[-1]:
        raise ValueError("The last dimension of X and Y must be the same.")
    # Save the original response
    # This corresponds to Y0 <- Y in Hoff's R code.
    # We need Y0 because Y will later be modified at missing
    # positions during the iterative fitting procedure.
    Y0 = Y.copy()
    # Replace missing Y values temporarily
    # Y0 still contains the original missing values.
    # Y is only a working copy used by the ALS algorithm.
    Y_work = Y.copy()
    Y_work[np.isnan(Y_work)] = 0.0
    # Determine tensor dimensions
    m = Y.shape
    p = X.shape
    # The final dimension is time. Therefore the number of coefficient matrices is:
    K = len(m) - 1
    # Initialize B
    if seed is None:
        # Deterministic initialization used by Hoff
        B0 = initialize_B(m, p)
        # Start B as a copy of B0.
        # We use copies because B will be modified.
        B = [Bk.copy() for Bk in B0]
    else:
        # Random initialization.
        rng = np.random.default_rng(seed)
        B = []
        scale = np.prod(p)
        for k in range(K):
            Bk = rng.normal(loc=0.0, scale=1.0, size=(m[k], p[k]))
            Bk = Bk / scale
            B.append(Bk)
    # Initial fitted values
    EY = tprod(X, B, modes=range(K))
    # Impute missing Y values
    missing = np.isnan(Y0)
    Y_work[missing] = EY[missing]
    # Initial MSE
    current_mse = mse(Y0, EY)
    old_mse = np.inf
    # ALS iterations
    i = 0
    with tqdm(total=imax, desc="ALS", unit="iter", dynamic_ncols=True, disable=not verbose) as pbar:
        while ((old_mse - current_mse) / current_mse > tol and i < imax):
            old_mse = current_mse
            # Randomize the order of the modes
            if seed is None:
                mode_order = np.random.permutation(K)
            else:
                mode_order = rng.permutation(K)
            # Update each B[k]
            for k in mode_order:
                # Transform X using all B matrices except B[k].
                other_modes = [j for j in range(K) if j != k]
                X_transformed = tprod(X,B,modes=other_modes)
                # Matricize X and Y.
                Xk = mat(X_transformed, k)
                Yk = mat(Y_work, k)
                # Solve: B[k] = Yk Xk'(Xk Xk')^{-1}
                Xt = Xk.T
                XXt = Xk @ Xt
                YXt = Yk @ Xt
                try:
                    B[k] = np.linalg.solve(XXt.T, YXt.T).T
                except np.linalg.LinAlgError:
                    # If Xk Xk' is singular or nearly singular, use the Moore-Penrose pseudoinverse.
                    B[k] = (YXt @ np.linalg.pinv(XXt))
            # Recalculate fitted values
            EY = tprod(X, B, modes=range(K))
            # Update missing observations
            Y_work[missing] = EY[missing]
            # Calculate the new MSE
            current_mse = mse(Y0, EY)
            # Increase iteration counter
            i += 1
            # Print convergence information
            improvement = ((old_mse - current_mse)/current_mse)
            pbar.update(1)
            pbar.set_postfix(MSE=f"{current_mse:.4e}", improvement=f"{improvement:.4e}", tol=f"{tol:.1e}")
    # Final information
    if verbose:
        if i >= imax:
            print(f"\nALS stopped because imax={imax} " 
                f"was reached.")
        else:
            print(f"\nALS converged after {i} iterations.")
        print(f"Final MSE = {current_mse:.8e}")
    # Return estimated coefficient matrices
    return B

In [28]:
Bhat =  mlm_als(Y, X_1w, tol=1e-10, imax=1000, verbose=True, seed=None)

ALS:   2%|▏         | 17/1000 [00:10<10:11,  1.61iter/s, MSE=9.0522e-01, improvement=5.4731e-11, tol=1.0e-10]


ALS converged after 17 iterations.
Final MSE = 9.05222071e-01
